# LeetCode 3245 — Alternating Groups III bằng Python

- Link: https://leetcode.com/problems/alternating-groups-iii/
- Độ khó: **Hard**
- Chủ đề: **"Cạnh xấu"** (hai ô kề cùng màu) chia vòng tròn thành các **đoạn xen kẽ** độ dài `L`; một đoạn đóng góp `max(0, L − s + 1)` nhóm cỡ `s` ⇒ hai **Fenwick tree** (đếm và tổng theo `L`) + tập có thứ tự các cạnh xấu (Fenwick tìm phần tử thứ `k`)

Skeleton trên LeetCode (Python3):

```python
class Solution:
    def numberOfAlternatingGroups(self, colors: List[int], queries: List[List[int]]) -> List[int]:
```

## 1. Mục tiêu học

1. Gọi cạnh `i` (giữa ô `i` và ô `i+1 mod n`) là **xấu** nếu hai ô cùng màu. Nhóm xen kẽ cỡ `s` bắt đầu tại ô `t` ⇔ `s − 1` cạnh `t, t+1, …, t+s−2` đều **tốt**.
2. Các cạnh xấu chia vòng tròn thành các **đoạn** ô liên tiếp; đoạn giữa hai cạnh xấu liên tiếp `p < q` gồm `q − p` ô (mod `n`). Đoạn `L` ô chứa `max(0, L − s + 1)` nhóm cỡ `s`.
3. Đáp án truy vấn cỡ `s` `= Σ_{L ≥ s} (L − s + 1) = (Σ_{L ≥ s} L) − (s − 1)·#{L ≥ s}` ⇒ hai Fenwick theo `L`.
4. Đổi màu ô `i` chỉ ảnh hưởng hai cạnh `i − 1` và `i` ⇒ thêm/bớt cạnh xấu = **tách/gộp** đoạn — cần cạnh xấu liền trước/sau (Fenwick tìm phần tử thứ `k`).
5. Trường hợp đặc biệt: **không có cạnh xấu** ⇒ cả vòng xen kẽ, đáp án `= n`.

## 2. Đề bài & ví dụ

Các ô đỏ (`0`)/xanh (`1`) xếp **vòng tròn**. **Nhóm xen kẽ** là một dãy ô liên tiếp (trên vòng) mà hai ô kề nhau trong nhóm khác màu. Xử lý truy vấn:
- `[1, size]`: đếm số nhóm xen kẽ có đúng `size` ô;
- `[2, index, color]`: đổi `colors[index] = color`.

**Ví dụ 1:** `colors = [0,1,1,0,1]`, `queries = [[2,1,0],[1,4]]` → `[2]`.
**Ví dụ 2:** `colors = [0,0,1,0,1,1]`, `queries = [[1,3],[2,3,0],[1,5]]` → `[2,0]`.

**Ràng buộc:** `4 ≤ n ≤ 5·10^4`; `1 ≤ q ≤ 5·10^4`; `3 ≤ size ≤ n − 1`.

In [1]:
from typing import List
import bisect

examples = [([0, 1, 1, 0, 1], [[2, 1, 0], [1, 4]], [2]), ([0, 0, 1, 0, 1, 1], [[1, 3], [2, 3, 0], [1, 5]], [2, 0])]
for c, q, exp in examples:
    print(c, q, '-> kỳ vọng', exp)

[0, 1, 1, 0, 1] [[2, 1, 0], [1, 4]] -> kỳ vọng [2]
[0, 0, 1, 0, 1, 1] [[1, 3], [2, 3, 0], [1, 5]] -> kỳ vọng [2, 0]


## 3. Kiến thức nền

### 3.1. Cạnh xấu và các đoạn

In [2]:
def bad_edges(c):
    n = len(c)
    return [i for i in range(n) if c[i] == c[(i + 1) % n]]

def runs_from(c):
    n = len(c); bs = bad_edges(c)
    if not bs:
        return None                                  # cả vòng xen kẽ
    return [((q - p) % n) or n for p, q in zip(bs, bs[1:] + bs[:1])]

c = [0, 0, 1, 0, 1, 1]
print('cạnh xấu:', bad_edges(c), '| độ dài các đoạn:', runs_from(c))

cạnh xấu: [0, 4] | độ dài các đoạn: [4, 2]


Với `[0,0,1,0,1,1]`: cạnh xấu `0` (ô 0–1) và `4` (ô 4–5); hai đoạn: ô `1..4` (4 ô) và ô `5, 0` (2 ô).

### 3.2. Đếm nhóm cỡ `s` trong các đoạn

Đoạn `L` ô có `L − s + 1` cửa sổ `s` ô (nếu `L ≥ s`). Không có cạnh xấu ⇒ mỗi ô là điểm bắt đầu của một nhóm cỡ `s` (với `s ≤ n − 1` không nhóm nào "bọc" trọn vòng) ⇒ `n` nhóm.

### 3.3. Fenwick đếm/tổng theo độ dài và Fenwick "phần tử thứ `k`"

In [3]:
class BIT:
    def __init__(self, n):
        self.n, self.t, self.LOG = n, [0] * (n + 1), n.bit_length()
    def add(self, i, v):
        i += 1
        while i <= self.n:
            self.t[i] += v; i += i & -i
    def pre(self, i):                    # tổng [0, i)
        r = 0
        while i > 0:
            r += self.t[i]; i -= i & -i
        return r
    def kth(self, k):                    # chỉ số (0-based) của phần tử thứ k (k >= 1)
        pos = 0
        for b in range(self.LOG, -1, -1):
            np_ = pos + (1 << b)
            if np_ <= self.n and self.t[np_] < k:
                pos = np_; k -= self.t[np_]
        return pos

b = BIT(10)
for x in [2, 5, 7]:
    b.add(x, 1)
print('phần tử thứ 1, 2, 3:', [b.kth(k) for k in (1, 2, 3)], '| số phần tử < 6:', b.pre(6))

phần tử thứ 1, 2, 3: [2, 5, 7] | số phần tử < 6: 2


## 4. Ý tưởng nền tảng / cách nghĩ trực giác

Ví dụ 2, truy vấn `[1, 3]`: đoạn dài `4` và `2` ⇒ `(4 − 3 + 1) + 0 = 2` ✓. Đổi `colors[3] = 0` (vốn đã là 0) ⇒ không đổi. Truy vấn `[1, 5]`: không đoạn nào dài `≥ 5` ⇒ `0` ✓.

Đổi màu một ô có thể: biến cạnh tốt thành xấu (**tách** một đoạn `L` thành hai đoạn `a + b = L`) hoặc ngược lại (**gộp** hai đoạn).

## 5. Cách 1 — Brute Force: kiểm tra mọi điểm bắt đầu

In [4]:
def brute(colors, queries):
    c = list(colors); n = len(c); out = []
    for q in queries:
        if q[0] == 1:
            s = q[1]
            out.append(sum(1 for st in range(n) if all(c[(st + t) % n] != c[(st + t + 1) % n] for t in range(s - 1))))
        else:
            c[q[1]] = q[2]
    return out

for c, q, exp in examples:
    print(brute(c, q), '| kỳ vọng', exp)

[2] | kỳ vọng [2]
[2, 0] | kỳ vọng [2, 0]


**Độ phức tạp:** `O(n · s)` mỗi truy vấn loại 1.

## 6. Ý tưởng thuật toán / bước đột phá

- `bad`: Fenwick trên vị trí cạnh `0..n−1` (1 = xấu) — cho phép tìm cạnh xấu liền trước/sau bằng `kth`.
- `cntL`, `sumL`: Fenwick trên độ dài `1..n` — số đoạn và tổng độ dài.
- Truy vấn `[1, s]`: nếu không có cạnh xấu ⇒ `n`; ngược lại `sumL[s..n] − (s − 1)·cntL[s..n]`.
- Đổi màu ô `i`: tính lại trạng thái cạnh `i − 1` và `i`; mỗi cạnh đổi trạng thái ⇒ tách/gộp đoạn với cạnh xấu liền trước `p` và liền sau `q`.

Mỗi thao tác `O(log n)`.

## 7. Các cách tiếp theo

### 7.1. Cách 2 — Tính lại các đoạn sau mỗi lần đổi màu (`O(n)` mỗi truy vấn)

In [5]:
def recompute_runs(colors, queries):
    c = list(colors); n = len(c); out = []
    for q in queries:
        if q[0] == 1:
            s = q[1]; runs = runs_from(c)
            out.append(n if runs is None else sum(L - s + 1 for L in runs if L >= s))
        else:
            c[q[1]] = q[2]
    return out

for c, q, exp in examples:
    print(recompute_runs(c, q), '| kỳ vọng', exp)

[2] | kỳ vọng [2]
[2, 0] | kỳ vọng [2, 0]


### 7.2. Cách 3 — Fenwick cho cạnh xấu + Fenwick cho độ dài đoạn (lời giải chính)

In [6]:
def fenwick_solution(colors, queries):
    c = list(colors); n = len(c)
    bad, cntL, sumL = BIT(n), BIT(n + 1), BIT(n + 1)
    isbad = [0] * n
    total = 0
    def run_add(L, sg):
        cntL.add(L, sg); sumL.add(L, sg * L)
    for i in range(n):
        if c[i] == c[(i + 1) % n]:
            isbad[i] = 1; bad.add(i, 1); total += 1
    bs = [i for i in range(n) if isbad[i]]
    for p, q in zip(bs, bs[1:] + bs[:1]):
        run_add(((q - p) % n) or n, 1)

    def neighbors(i):                    # cạnh xấu liền trước và liền sau i (vòng), i không tính
        tot = bad.pre(n)
        k = bad.pre(i)
        prv = bad.kth(k) if k >= 1 else bad.kth(tot)
        k2 = bad.pre(i + 1)
        nxt = bad.kth(k2 + 1) if k2 < tot else bad.kth(1)
        return prv, nxt

    def toggle(i, make_bad):
        nonlocal total
        if make_bad:
            if total == 0:
                run_add(n, 1)
            else:
                p, q = neighbors(i)
                run_add(((q - p) % n) or n, -1)
                run_add(((i - p) % n) or n, 1); run_add(((q - i) % n) or n, 1)
            bad.add(i, 1); total += 1
        else:
            bad.add(i, -1); total -= 1
            if total == 0:
                run_add(n, -1)
            else:
                p, q = neighbors(i)
                run_add(((i - p) % n) or n, -1); run_add(((q - i) % n) or n, -1)
                run_add(((q - p) % n) or n, 1)

    out = []
    for qr in queries:
        if qr[0] == 1:
            s = qr[1]
            if total == 0:
                out.append(n)
            else:
                out.append((sumL.pre(n + 1) - sumL.pre(s)) - (s - 1) * (cntL.pre(n + 1) - cntL.pre(s)))
        else:
            i, col = qr[1], qr[2]
            if c[i] == col:
                continue
            c[i] = col
            for e in ((i - 1) % n, i):
                nb_ = 1 if c[e] == c[(e + 1) % n] else 0
                if nb_ != isbad[e]:
                    isbad[e] = nb_
                    toggle(e, nb_ == 1)
    return out

for c, q, exp in examples:
    print(fenwick_solution(c, q), '| kỳ vọng', exp)

[2] | kỳ vọng [2]
[2, 0] | kỳ vọng [2, 0]


Khi chỉ còn **một** cạnh xấu `p`, đoạn duy nhất dài `n` ô (từ `p+1` vòng về `p`) — biểu thức `((q − p) % n) or n` với `q = p` cho đúng `n`.

### 7.3. Cách 4 — Tập có thứ tự (SortedList) thay cho Fenwick vị trí

Lưu các cạnh xấu trong danh sách đã sắp; tìm liền trước/sau bằng `bisect`. Dùng `sortedcontainers` nếu có (LeetCode có sẵn), ngược lại dùng `list` + `bisect.insort` (chèn `O(n)` nhưng hằng số nhỏ).

In [7]:
try:
    from sortedcontainers import SortedList
except ImportError:
    class SortedList(list):
        def __init__(self, it=()):
            super().__init__(sorted(it))
        def add(self, v):
            bisect.insort(self, v)
        def remove(self, v):
            del self[bisect.bisect_left(self, v)]
        def bisect_left(self, v):
            return bisect.bisect_left(self, v)
        def bisect_right(self, v):
            return bisect.bisect_right(self, v)

def sortedlist_solution(colors, queries):
    c = list(colors); n = len(c)
    S = SortedList(i for i in range(n) if c[i] == c[(i + 1) % n])
    cntL, sumL = BIT(n + 1), BIT(n + 1)
    def run_add(L, sg):
        cntL.add(L, sg); sumL.add(L, sg * L)
    lst = list(S)
    for p, q in zip(lst, lst[1:] + lst[:1]):
        run_add(((q - p) % n) or n, 1)
    def around(i):
        k = S.bisect_left(i)
        return S[k - 1], S[k % len(S)]
    out = []
    for qr in queries:
        if qr[0] == 1:
            s = qr[1]
            out.append(n if not S else (sumL.pre(n + 1) - sumL.pre(s)) - (s - 1) * (cntL.pre(n + 1) - cntL.pre(s)))
            continue
        i, col = qr[1], qr[2]
        if c[i] == col:
            continue
        c[i] = col
        for e in ((i - 1) % n, i):
            now = c[e] == c[(e + 1) % n]
            k = S.bisect_left(e); was = k < len(S) and S[k] == e
            if now == was:
                continue
            if now:
                if not S:
                    run_add(n, 1)
                else:
                    p, q = around(e)
                    run_add(((q - p) % n) or n, -1); run_add(((e - p) % n) or n, 1); run_add(((q - e) % n) or n, 1)
                S.add(e)
            else:
                S.remove(e)
                if not S:
                    run_add(n, -1)
                else:
                    p, q = around(e)
                    run_add(((e - p) % n) or n, -1); run_add(((q - e) % n) or n, -1); run_add(((q - p) % n) or n, 1)
    return out

for c, q, exp in examples:
    print(sortedlist_solution(c, q), '| kỳ vọng', exp)

[2] | kỳ vọng [2]
[2, 0] | kỳ vọng [2, 0]


`S[k − 1]` với `k = 0` là phần tử **cuối** (Python cho phép chỉ số âm) — đúng ý nghĩa "liền trước theo vòng".

## 8. TRACE từng bước

Trace cách 2 (dễ thấy các đoạn) trên ví dụ 1.

In [8]:
def trace(colors, queries):
    c = list(colors); n = len(c)
    print('ban đầu:', c, '| cạnh xấu', bad_edges(c), '| đoạn', runs_from(c))
    for q in queries:
        if q[0] == 2:
            c[q[1]] = q[2]
            print(f'đổi màu ô {q[1]} -> {q[2]}: {c} | cạnh xấu {bad_edges(c)} | đoạn {runs_from(c)}')
        else:
            runs = runs_from(c); s = q[1]
            print(f'đếm cỡ {s}:', n if runs is None else [max(0, L - s + 1) for L in runs], '->',
                  n if runs is None else sum(max(0, L - s + 1) for L in runs))
    print('vét cạn:', brute(colors, queries))

trace(*examples[0][:2])

ban đầu: [0, 1, 1, 0, 1] | cạnh xấu [1] | đoạn [5]
đổi màu ô 1 -> 0: [0, 0, 1, 0, 1] | cạnh xấu [0] | đoạn [5]
đếm cỡ 4: [2] -> 2
vét cạn: [2]


## 9. Code nộp LeetCode

In [9]:
class Solution:
    def numberOfAlternatingGroups(self, colors: List[int], queries: List[List[int]]) -> List[int]:
        c = list(colors)
        n = len(c)

        class Fen:
            __slots__ = ('n', 't', 'LOG')
            def __init__(self, n):
                self.n = n; self.t = [0] * (n + 1); self.LOG = n.bit_length()
            def add(self, i, v):
                i += 1
                while i <= self.n:
                    self.t[i] += v; i += i & -i
            def pre(self, i):
                r = 0
                while i > 0:
                    r += self.t[i]; i -= i & -i
                return r
            def kth(self, k):
                pos = 0
                for b in range(self.LOG, -1, -1):
                    q = pos + (1 << b)
                    if q <= self.n and self.t[q] < k:
                        pos = q; k -= self.t[q]
                return pos

        bad, cnt, sm = Fen(n), Fen(n + 1), Fen(n + 1)
        isbad = [0] * n
        total = 0
        for i in range(n):
            if c[i] == c[(i + 1) % n]:
                isbad[i] = 1
                bad.add(i, 1)
                total += 1

        def run(L, sg):
            cnt.add(L, sg); sm.add(L, sg * L)

        bs = [i for i in range(n) if isbad[i]]
        for p, q in zip(bs, bs[1:] + bs[:1]):
            run(((q - p) % n) or n, 1)

        def nbrs(i):
            tot = total
            k = bad.pre(i)
            p = bad.kth(k) if k >= 1 else bad.kth(tot)
            k2 = bad.pre(i + 1)
            q = bad.kth(k2 + 1) if k2 < tot else bad.kth(1)
            return p, q

        ans = []
        for qr in queries:
            if qr[0] == 1:
                s = qr[1]
                if total == 0:
                    ans.append(n)
                else:
                    ans.append(sm.pre(n + 1) - sm.pre(s) - (s - 1) * (cnt.pre(n + 1) - cnt.pre(s)))
                continue
            i, col = qr[1], qr[2]
            if c[i] == col:
                continue
            c[i] = col
            for e in ((i - 1) % n, i):
                now = 1 if c[e] == c[(e + 1) % n] else 0
                if now == isbad[e]:
                    continue
                isbad[e] = now
                if now:
                    if total == 0:
                        run(n, 1)
                    else:
                        p, q = nbrs(e)
                        run(((q - p) % n) or n, -1)
                        run(((e - p) % n) or n, 1)
                        run(((q - e) % n) or n, 1)
                    bad.add(e, 1); total += 1
                else:
                    bad.add(e, -1); total -= 1
                    if total == 0:
                        run(n, -1)
                    else:
                        p, q = nbrs(e)
                        run(((e - p) % n) or n, -1)
                        run(((q - e) % n) or n, -1)
                        run(((q - p) % n) or n, 1)
        return ans

for c, q, exp in examples:
    got = Solution().numberOfAlternatingGroups(c, q)
    print(got, 'OK' if got == exp else 'SAI')

[2] OK
[2, 0] OK


## 10. Vì sao thuật toán đúng

1. **Nhóm ⇔ cửa sổ không chứa cạnh xấu:** nhóm cỡ `s` bắt đầu tại `t` dùng các cạnh `t .. t+s−2`; hợp lệ ⇔ không cạnh nào xấu. Nếu có cạnh xấu, mỗi cửa sổ hợp lệ nằm gọn trong một đoạn giữa hai cạnh xấu liên tiếp; đoạn `L` ô có đúng `L − s + 1` cửa sổ (`L ≥ s`).
2. **Không có cạnh xấu:** mọi `t` đều hợp lệ (`s ≤ n − 1` nên cửa sổ không tự chồng lấn vòng) ⇒ `n`.
3. **Cập nhật:** đổi màu ô `i` chỉ thay đổi cạnh `i − 1` và `i`. Thêm cạnh xấu `e` vào giữa `p` và `q` (liền trước/sau) tách đoạn `(q − p)` thành `(e − p)` và `(q − e)`; bớt thì gộp ngược lại — giữ đúng đa tập độ dài đoạn.
4. **Fenwick `kth`:** trả về cạnh xấu thứ `k` theo vị trí ⇒ liền trước = thứ `#(< i)`, liền sau = thứ `#(≤ i) + 1` (vòng về thứ 1 / thứ cuối khi tràn).

## 11. Phân tích Time / Space Complexity

| Cách | Time mỗi truy vấn | Space |
|---|---|---|
| 1. Vét cạn | `O(n · s)` | `O(n)` |
| 2. Tính lại đoạn | `O(n)` | `O(n)` |
| 3. Fenwick + Fenwick | `O(log n)` | `O(n)` |
| 4. SortedList + Fenwick | `O(log n)` (`O(n)` nếu dùng list thường) | `O(n)` |

## 12. Bộ test

### 12.1. Ví dụ đề + ca biên

In [10]:
impls = {'recompute': recompute_runs, 'fenwick': fenwick_solution, 'sortedlist': sortedlist_solution,
         'submit': lambda c, q: Solution().numberOfAlternatingGroups(c, q)}
edge = [
    ([0, 1, 0, 1], [[1, 3]], [4]),                               # cả vòng xen kẽ
    ([0, 0, 0, 0], [[1, 3], [2, 1, 1], [1, 3]], None),
    ([0, 1, 0, 1, 0, 1], [[2, 0, 1], [1, 5], [2, 0, 0], [1, 5]], None),   # tạo rồi xóa cạnh xấu
    ([1, 1, 0, 1], [[1, 3], [2, 0, 0], [1, 3]], None),
]
for c, q, exp in examples + edge:
    e = brute(c, q)
    if exp is not None:
        assert e == exp, (c, q, e)
    for name, f in impls.items():
        assert f(c, q) == e, (name, c, q)
print('Ví dụ đề và ca biên: tất cả đúng.')

Ví dụ đề và ca biên: tất cả đúng.


### 12.2. Đối chiếu ngẫu nhiên với vét cạn

In [11]:
import random
rng = random.Random(3245)
cnt = 0
for _ in range(1500):
    n = rng.randint(4, 10)
    c = [rng.randint(0, 1) for _ in range(n)]
    qs = [[1, rng.randint(3, n - 1)] if rng.random() < 0.5 else [2, rng.randrange(n), rng.randint(0, 1)]
          for _ in range(rng.randint(1, 12))]
    exp = brute(c, qs)
    for name, f in impls.items():
        assert f(c, qs) == exp, (name, c, qs)
    cnt += 1
print('Khớp vét cạn trên', cnt, 'bộ test.')

Khớp vét cạn trên 1500 bộ test.


### 12.3. Ca lớn `n = q = 5·10^4`

In [12]:
import time
N = 5 * 10 ** 4
for name, c in [('ngẫu nhiên', [rng.randint(0, 1) for _ in range(N)]),
                ('gần xen kẽ', [i % 2 for i in range(N)])]:
    qs = [[1, rng.randint(3, N - 1)] if rng.random() < 0.5 else [2, rng.randrange(N), rng.randint(0, 1)] for _ in range(N)]
    t = time.time(); a = Solution().numberOfAlternatingGroups(c, qs); t1 = time.time() - t
    t = time.time(); b = sortedlist_solution(c, qs); t2 = time.time() - t
    sub = qs[:300]
    assert a == b and recompute_runs(c, sub) == Solution().numberOfAlternatingGroups(c, sub)
    print(f"{name:12s}: nộp {t1:.2f}s | SortedList {t2:.2f}s | 300 truy vấn đầu khớp cách 2")

ngẫu nhiên  : nộp 0.75s | SortedList 0.52s | 300 truy vấn đầu khớp cách 2


gần xen kẽ  : nộp 0.46s | SortedList 0.42s | 300 truy vấn đầu khớp cách 2


## 13. Các lỗi người mới hay gặp

### Lỗi 1 — Quên tính vòng (coi như dãy thẳng)

In [13]:
def wrong_linear(colors, queries):
    c = list(colors); n = len(c); out = []
    for q in queries:
        if q[0] == 1:
            s = q[1]
            out.append(sum(1 for st in range(n - s + 1) if all(c[st + t] != c[st + t + 1] for t in range(s - 1))))   # SAI
        else:
            c[q[1]] = q[2]
    return out

bad = ([1, 0, 0, 1, 0], [[1, 3]])
print('sai :', wrong_linear(*bad), '| đúng:', Solution().numberOfAlternatingGroups(*bad))

sai : [1] | đúng: [3]


**Vì sao sai:** ô đầu và ô cuối kề nhau; nhóm có thể "bọc" qua ranh giới — với `[1,0,0,1,0]`, các nhóm `(3,4,0)` = `1,0,1` và `(4,0,1)` = `0,1,0` chỉ tồn tại trên vòng.

### Lỗi 2 — Không xét riêng trường hợp không có cạnh xấu

In [14]:
def wrong_no_bad_case(colors, queries):
    c = list(colors); n = len(c); out = []
    for q in queries:
        if q[0] == 1:
            s = q[1]; runs = runs_from(c) or [n]                       # SAI: coi như một đoạn dài n
            out.append(sum(L - s + 1 for L in runs if L >= s))
        else:
            c[q[1]] = q[2]
    return out

print('sai :', wrong_no_bad_case([0, 1, 0, 1], [[1, 3]]), '| đúng:', Solution().numberOfAlternatingGroups([0, 1, 0, 1], [[1, 3]]))

sai : [2] | đúng: [4]


**Vì sao sai:** không có cạnh xấu thì vòng **liên tục**, mọi ô là điểm bắt đầu ⇒ `n` (không phải `n − s + 1`).

### Lỗi 3 — Đổi màu ô `i` chỉ cập nhật cạnh `i` (quên cạnh `i − 1`)

In [15]:
def wrong_one_edge(colors, queries):
    c = list(colors); n = len(c)
    isbad = [int(c[i] == c[(i + 1) % n]) for i in range(n)]
    out = []
    for q in queries:
        if q[0] == 1:
            s = q[1]; bs = [i for i in range(n) if isbad[i]]
            if not bs:
                out.append(n); continue
            runs = [((b - a) % n) or n for a, b in zip(bs, bs[1:] + bs[:1])]
            out.append(sum(L - s + 1 for L in runs if L >= s))
        else:
            c[q[1]] = q[2]
            i = q[1]
            isbad[i] = int(c[i] == c[(i + 1) % n])                      # SAI: thiếu cạnh i - 1
    return out

print('sai :', wrong_one_edge(*examples[0][:2]), '| đúng:', Solution().numberOfAlternatingGroups(*examples[0][:2]))

sai : [5] | đúng: [2]


**Vì sao sai:** ô `i` thuộc hai cạnh: `(i − 1, i)` và `(i, i + 1)`.

### Lỗi 4 — Cộng `L − s + 1` cho **mọi** đoạn (kể cả `L < s`)

In [16]:
def wrong_negative(colors, queries):
    c = list(colors); n = len(c); out = []
    for q in queries:
        if q[0] == 1:
            s = q[1]; runs = runs_from(c)
            out.append(n if runs is None else sum(L - s + 1 for L in runs))      # SAI
        else:
            c[q[1]] = q[2]
    return out

print('sai :', wrong_negative(*examples[1][:2]), '| đúng:', Solution().numberOfAlternatingGroups(*examples[1][:2]))

sai : [2, -2] | đúng: [2, 0]


**Vì sao sai:** đoạn ngắn hơn `s` đóng góp **0**, không phải số âm — vì vậy Fenwick chỉ lấy tổng trên `L ≥ s`.

## 14. Mẹo làm bài nhanh & nhận dạng pattern

- "Đếm cửa sổ thỏa tính chất cục bộ giữa hai phần tử kề" ⇒ đánh dấu các **chỗ hỏng**, đếm theo đoạn giữa các chỗ hỏng.
- Truy vấn theo ngưỡng độ dài ⇒ Fenwick theo độ dài (đếm + tổng).
- Cần "liền trước / liền sau" trong tập động ⇒ Fenwick `kth` hoặc SortedList.
- Vòng tròn: cẩn thận trường hợp **0 chỗ hỏng** và **1 chỗ hỏng**.

## 15. Biến thể của bài

1. **LC 3206/3208 (bản I/II):** không có cập nhật, cỡ nhóm cố định — một lần duyệt.
2. Dãy **thẳng** thay vì vòng ⇒ hai đầu dãy đóng vai "cạnh xấu".
3. Nhiều màu, điều kiện "hai ô kề khác màu" — giữ nguyên lời giải.
4. Truy vấn đếm nhóm có cỡ **trong đoạn** `[s1, s2]` ⇒ hiệu hai truy vấn "cỡ ≥".

In [17]:
# Biến thể 2: dãy thẳng (không vòng), có cập nhật
def linear_variant(colors, queries):
    c = list(colors); n = len(c); out = []
    for q in queries:
        if q[0] == 1:
            s = q[1]; runs = []; L = 1
            for i in range(1, n):
                if c[i] != c[i - 1]:
                    L += 1
                else:
                    runs.append(L); L = 1
            runs.append(L)
            out.append(sum(x - s + 1 for x in runs if x >= s))
        else:
            c[q[1]] = q[2]
    return out

rng = random.Random(15)
for _ in range(300):
    n = rng.randint(4, 9); c = [rng.randint(0, 1) for _ in range(n)]
    qs = [[1, rng.randint(3, n - 1)] if rng.random() < 0.5 else [2, rng.randrange(n), rng.randint(0, 1)] for _ in range(6)]
    assert linear_variant(c, qs) == wrong_linear(c, qs)
print('Biến thể dãy thẳng = "lỗi 1" của bài gốc (đúng cho đề không vòng).')

Biến thể dãy thẳng = "lỗi 1" của bài gốc (đúng cho đề không vòng).


## 16. So sánh các cách làm

| Cách | Ý tưởng | Time / truy vấn | Ghi chú |
|---|---|---|---|
| 1 | Vét cạn | `O(n s)` | Oracle |
| 2 | Tính lại đoạn | `O(n)` | Dễ viết |
| 3 | Fenwick vị trí + Fenwick độ dài | `O(log n)` | **Nộp bài** |
| 4 | SortedList + Fenwick độ dài | `O(log n)` | Cần thư viện |

## 17. 5 bài tập vận dụng (từ dễ đến khó)

**Bài 1 (đoán output):** `colors = [0,1,0,1,0,1]`, `[[1, 4]]`.

**Bài 2 (tính tay):** `colors = [0,0,1,1]`, `[[1, 3]]`.

**Bài 3 (điền TODO):** biến thể 4 — đếm nhóm có cỡ trong `[s1, s2]`.

**Bài 4 (tự viết lại):** cỡ lớn nhất của nhóm xen kẽ hiện có (sau mỗi cập nhật).

**Bài 5 (mini challenge):** hỗ trợ thêm truy vấn "đảo màu toàn bộ" (mọi ô `0 ↔ 1`) — đáp án không đổi? Chứng minh bằng code.

In [18]:
def range_size_todo(colors, queries):
    # TODO
    raise NotImplementedError

def longest_group_todo(colors, queries):
    # TODO
    raise NotImplementedError

def flip_all_invariant_todo(colors, queries):
    # TODO
    raise NotImplementedError

print('Đã định nghĩa các hàm TODO (không gọi).')

Đã định nghĩa các hàm TODO (không gọi).


## 18. Gợi ý 3 mức cho bài tập

**Bài 1.** (1) Không cạnh xấu. (2) Mọi ô bắt đầu một nhóm. (3) `[6]`.

**Bài 2.** (1) Cạnh xấu `0` (ô 0–1) và `2` (ô 2–3). (2) Hai đoạn dài 2. (3) `[0]`.

**Bài 3.** (1) `F(s) =` số nhóm cỡ đúng `s`. (2) Tổng `F(s)` cho `s ∈ [s1, s2]` = hiệu của tổng tích lũy theo `L`… (3) Viết bằng Fenwick bậc hai hoặc lặp `s` (nhỏ).

**Bài 4.** (1) Đoạn dài nhất `L*`. (2) Không cạnh xấu ⇒ `n − 1` (cỡ tối đa cho phép). (3) Fenwick độ dài + tìm `L` lớn nhất có đếm > 0.

**Bài 5.** (1) Đảo màu mọi ô không đổi quan hệ "cùng màu / khác màu" giữa hai ô kề. (2) Cạnh xấu giữ nguyên. (3) Đáp án mọi truy vấn không đổi.

## 19. Đáp án tham khảo cho bài tập

In [19]:
# Bài 1, 2, 3
assert Solution().numberOfAlternatingGroups([0, 1, 0, 1, 0, 1], [[1, 4]]) == [6]
assert Solution().numberOfAlternatingGroups([0, 0, 1, 1], [[1, 3]]) == [0]

def range_size(colors, queries):
    c = list(colors); n = len(c); out = []
    for q in queries:
        if q[0] == 1:
            s1, s2 = q[1], q[2]
            out.append(sum(Solution().numberOfAlternatingGroups(c, [[1, s]])[0] for s in range(s1, s2 + 1)))
        else:
            c[q[1]] = q[2]
    return out

r_ = random.Random(3)
for _ in range(150):
    n = r_.randint(4, 8); c = [r_.randint(0, 1) for _ in range(n)]
    a = r_.randint(3, n - 1); b = r_.randint(a, n - 1)
    assert range_size(c, [[1, a, b]]) == [sum(brute(c, [[1, s]])[0] for s in range(a, b + 1))]
print('Bài 1: [6] | Bài 2: [0] | Bài 3 khớp vét cạn.')

Bài 1: [6] | Bài 2: [0] | Bài 3 khớp vét cạn.


In [20]:
# Bài 4, 5
def longest_group(colors, queries):
    c = list(colors); n = len(c); out = []
    for q in queries:
        if q[0] == 2:
            c[q[1]] = q[2]
        runs = runs_from(c)
        out.append(n - 1 if runs is None else min(max(runs), n - 1))
    return out

def flip_all_invariant(colors, queries):
    flipped = [1 - x for x in colors]
    fq = [q if q[0] == 1 else [2, q[1], 1 - q[2]] for q in queries]
    return Solution().numberOfAlternatingGroups(colors, queries) == Solution().numberOfAlternatingGroups(flipped, fq)

r_ = random.Random(4)
for _ in range(200):
    n = r_.randint(4, 9); c = [r_.randint(0, 1) for _ in range(n)]
    qs = [[1, r_.randint(3, n - 1)] if r_.random() < 0.5 else [2, r_.randrange(n), r_.randint(0, 1)] for _ in range(6)]
    cc = list(c); exp = []
    for q in qs:
        if q[0] == 2:
            cc[q[1]] = q[2]
        exp.append(max(s for s in range(1, n) if brute(cc, [[1, s]])[0] > 0))
    assert longest_group(c, qs) == exp
    assert flip_all_invariant(c, qs)
print('Bài 4 khớp vét cạn; Bài 5: đảo màu toàn bộ không đổi đáp án.')

Bài 4 khớp vét cạn; Bài 5: đảo màu toàn bộ không đổi đáp án.


## 20. Template nhớ nhanh + Checklist tự kiểm tra trước khi submit

```python
cạnh xấu e: c[e] == c[(e+1) % n];  đoạn giữa p < q (liên tiếp): (q − p) % n or n ô
truy vấn s: không cạnh xấu -> n;  ngược lại Σ_{L≥s} L − (s−1)·#{L≥s}   (Fenwick cnt, sum)
đổi ô i: xét cạnh i−1 và i; thêm cạnh e giữa p, q: −(q−p), +(e−p), +(q−e); bớt: ngược lại
```

**Checklist:**

- [ ] Vòng tròn (chỉ số mod `n`)?
- [ ] 0 cạnh xấu ⇒ `n`; 1 cạnh xấu ⇒ một đoạn dài `n`?
- [ ] Cập nhật **cả hai** cạnh quanh ô đổi màu?
- [ ] Chỉ cộng đoạn `L ≥ s`?
- [ ] Bỏ qua cập nhật không đổi màu?